# Improved H100 Image Ensemble

This is the self-contained Google Colab notebook. Every part of the complete
PyTorch training pipeline is visible and executable directly in the notebook
cells. It trains ConvNeXt, Swin, and two EVA02 variants.

Only the image dataset must already exist in Google Drive:

```text
MyDrive/Kaggle/train/<class number>/*.jpg
MyDrive/Kaggle/test/*.jpg
```

Choose an H100 or A100 runtime before running the cells. A complete run is
computationally expensive.


## 1. Install Dependencies

The notebook uses PyTorch from the Colab runtime and installs the additional
libraries needed by the project.


In [ ]:
!pip -q install timm scikit-learn pandas Pillow


## 2. Mount Google Drive

Run this cell directly inside the browser version of Google Colab. If Google
returns an authentication `400` error, disconnect the runtime, allow Google
authentication pop-ups, reconnect, and run the cell again.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 3. Configure PyTorch Memory Allocation

Expandable CUDA segments can reduce failures caused by reserved-memory
fragmentation during long multi-model runs.


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


## 4. Define the Training Pipeline

The following cells contain the full implementation directly. No Python package
files or source-code upload are required.


### Configuration


In [ ]:
"""Configuration for training and blending the image models."""


from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any


@dataclass(frozen=True)
class VariantConfig:
    """One independently trained model variant."""

    name: str
    family: str
    model_patterns: tuple[str, ...]
    image_size: int
    batch_size: int
    accum_steps: int
    learning_rate: float
    weight_decay: float
    drop_rate: float
    drop_path_rate: float
    mixup_alpha: float
    cutmix_alpha: float
    mixup_probability: float
    label_smoothing: float
    random_erasing_probability: float
    rotation_degrees: int
    color_jitter_strength: float
    crop_minimum: float
    warmup_epochs: int
    patience: int
    seed_offset: int = 0
    backbone_lr_multiplier: float = 0.35
    gradient_checkpointing: bool = True
    use_ema: bool = False
    ema_decay: float = 0.9998
    enabled: bool = True


@dataclass(frozen=True)
class ProjectConfig:
    """Settings shared across the full experiment."""

    train_dir: Path
    test_dir: Path
    output_dir: Path
    num_classes: int = 100
    num_folds: int = 4
    final_epochs: int = 120
    prediction_batch_size: int = 8
    num_workers: int = 2
    pin_memory: bool = False
    persistent_workers: bool = False
    prefetch_factor: int = 1
    seed: int = 42
    use_amp: bool = True
    automatically_balance_classes: bool = True
    imbalance_ratio_threshold: float = 1.5
    fold_weight_temperature: float = 0.08
    blend_objective: str = "log_loss"
    blend_search_iterations: int = 4000
    tta_candidates: tuple[str, ...] = ("none", "horizontal_flip")
    variants: tuple[VariantConfig, ...] = field(default_factory=tuple)

    def to_dict(self) -> dict[str, Any]:
        data = asdict(self)
        for key in ("train_dir", "test_dir", "output_dir"):
            data[key] = str(data[key])
        return data


def default_colab_config(
    kaggle_dir: str | Path = "/content/drive/MyDrive/Kaggle",
) -> ProjectConfig:
    """Return the recommended full H100/A100 experiment configuration."""

    kaggle_dir = Path(kaggle_dir)
    return ProjectConfig(
        train_dir=kaggle_dir / "train",
        test_dir=kaggle_dir / "test",
        output_dir=kaggle_dir / "improved_oof_ensemble",
        variants=(
            VariantConfig(
                name="convnext_large",
                family="convnext",
                model_patterns=(
                    "convnext_large.fb_in22k_ft_in1k",
                    "convnext_large",
                ),
                image_size=384,
                batch_size=16,
                accum_steps=4,
                learning_rate=1.73e-4,
                weight_decay=0.089,
                drop_rate=0.141,
                drop_path_rate=0.186,
                mixup_alpha=0.15,
                cutmix_alpha=0.50,
                mixup_probability=0.68,
                label_smoothing=0.08,
                random_erasing_probability=0.25,
                rotation_degrees=8,
                color_jitter_strength=0.23,
                crop_minimum=0.59,
                warmup_epochs=5,
                patience=12,
            ),
            VariantConfig(
                name="swin_base",
                family="swin",
                model_patterns=(
                    "swin_base_patch4_window12_384.ms_in22k_ft_in1k",
                    "swin_base_patch4_window12_384",
                    "swin_base",
                ),
                image_size=384,
                batch_size=12,
                accum_steps=4,
                learning_rate=5.93e-5,
                weight_decay=0.072,
                drop_rate=0.155,
                drop_path_rate=0.165,
                mixup_alpha=0.12,
                cutmix_alpha=0.70,
                mixup_probability=0.65,
                label_smoothing=0.08,
                random_erasing_probability=0.21,
                rotation_degrees=6,
                color_jitter_strength=0.28,
                crop_minimum=0.66,
                warmup_epochs=6,
                patience=12,
            ),
            VariantConfig(
                name="eva02_large_seed_a",
                family="eva02",
                model_patterns=(
                    "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
                    "eva02_large_patch14_448",
                    "eva02_large",
                ),
                image_size=448,
                batch_size=6,
                accum_steps=16,
                learning_rate=9.13e-5,
                weight_decay=0.075,
                drop_rate=0.166,
                drop_path_rate=0.175,
                mixup_alpha=0.03,
                cutmix_alpha=0.50,
                mixup_probability=0.545,
                label_smoothing=0.04,
                random_erasing_probability=0.30,
                rotation_degrees=6,
                color_jitter_strength=0.17,
                crop_minimum=0.59,
                warmup_epochs=4,
                patience=20,
                seed_offset=0,
            ),
            VariantConfig(
                name="eva02_large_seed_b",
                family="eva02",
                model_patterns=(
                    "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
                    "eva02_large_patch14_448",
                    "eva02_large",
                ),
                image_size=448,
                batch_size=6,
                accum_steps=16,
                learning_rate=8.50e-5,
                weight_decay=0.075,
                drop_rate=0.166,
                drop_path_rate=0.175,
                mixup_alpha=0.03,
                cutmix_alpha=0.50,
                mixup_probability=0.545,
                label_smoothing=0.04,
                random_erasing_probability=0.30,
                rotation_degrees=6,
                color_jitter_strength=0.17,
                crop_minimum=0.62,
                warmup_epochs=4,
                patience=20,
                seed_offset=10_000,
            ),
        ),
    )


### OOF Ensembling


In [ ]:
"""Small, dependency-light utilities for validation-driven ensembling."""


from dataclasses import asdict, dataclass
from typing import Mapping, Sequence

import numpy as np

EPSILON = 1e-12


@dataclass(frozen=True)
class BlendResult:
    """The selected blend method and its out-of-fold measurements."""

    names: tuple[str, ...]
    weights: tuple[float, ...]
    mode: str
    accuracy: float
    log_loss: float
    objective: str

    def to_dict(self) -> dict[str, object]:
        return asdict(self)


def normalize_probabilities(probabilities: np.ndarray) -> np.ndarray:
    """Clip numerical noise and make each row sum to one."""

    probabilities = np.asarray(probabilities)
    if probabilities.ndim != 2:
        raise ValueError("Expected a two-dimensional probability matrix.")
    dtype = np.float32 if probabilities.dtype == np.float32 else np.float64
    probabilities = probabilities.astype(dtype, copy=False)
    probabilities = np.clip(probabilities, EPSILON, None)
    return probabilities / probabilities.sum(axis=1, keepdims=True)


def classification_accuracy(probabilities: np.ndarray, labels: np.ndarray) -> float:
    probabilities = normalize_probabilities(probabilities)
    labels = np.asarray(labels)
    return float(np.mean(probabilities.argmax(axis=1) == labels))


def multiclass_log_loss(probabilities: np.ndarray, labels: np.ndarray) -> float:
    probabilities = normalize_probabilities(probabilities)
    labels = np.asarray(labels, dtype=np.int64)
    if probabilities.shape[0] != labels.shape[0]:
        raise ValueError("Prediction and label counts do not match.")
    chosen = probabilities[np.arange(labels.shape[0]), labels]
    return float(-np.mean(np.log(np.clip(chosen, EPSILON, 1.0))))


def blend_predictions(
    prediction_sets: Sequence[np.ndarray],
    weights: Sequence[float],
    mode: str = "probabilities",
) -> np.ndarray:
    """Blend probabilities arithmetically or as log-probability scores."""

    if not prediction_sets:
        raise ValueError("At least one prediction matrix is required.")

    weights_array = np.asarray(weights, dtype=np.float64)
    if weights_array.shape != (len(prediction_sets),):
        raise ValueError("There must be one weight per prediction matrix.")
    if np.any(weights_array < 0) or weights_array.sum() <= 0:
        raise ValueError("Blend weights must be non-negative with a positive sum.")
    weights_array = weights_array / weights_array.sum()

    matrices = [normalize_probabilities(matrix) for matrix in prediction_sets]
    if len({matrix.shape for matrix in matrices}) != 1:
        raise ValueError("All prediction matrices must have the same shape.")

    stacked = np.stack(matrices, axis=0)
    if mode == "probabilities":
        blended = np.tensordot(weights_array, stacked, axes=(0, 0))
    elif mode == "logits":
        log_scores = np.tensordot(weights_array, np.log(stacked), axes=(0, 0))
        log_scores -= log_scores.max(axis=1, keepdims=True)
        blended = np.exp(log_scores)
    else:
        raise ValueError(f"Unknown blend mode: {mode}")

    return normalize_probabilities(blended)


def fold_weights_from_losses(
    losses: Sequence[float],
    temperature: float = 0.08,
) -> np.ndarray:
    """Give lower-loss folds more influence while retaining fold diversity."""

    losses_array = np.asarray(losses, dtype=np.float64)
    if losses_array.ndim != 1 or losses_array.size == 0:
        raise ValueError("At least one fold loss is required.")
    if temperature <= 0:
        raise ValueError("Temperature must be positive.")

    scores = -(losses_array - losses_array.min()) / temperature
    scores -= scores.max()
    weights = np.exp(scores)
    return weights / weights.sum()


def _score_candidate(
    probabilities: np.ndarray,
    labels: np.ndarray,
    objective: str,
) -> tuple[float, float]:
    accuracy = classification_accuracy(probabilities, labels)
    log_loss = multiclass_log_loss(probabilities, labels)
    if objective == "log_loss":
        return log_loss, -accuracy
    if objective == "accuracy":
        return -accuracy, log_loss
    raise ValueError(f"Unsupported blend objective: {objective}")


def _candidate_weights(num_models: int, iterations: int, seed: int) -> list[np.ndarray]:
    rng = np.random.default_rng(seed)
    candidates = [np.full(num_models, 1.0 / num_models)]
    candidates.extend(np.eye(num_models))
    candidates.extend(rng.dirichlet(np.ones(num_models), size=iterations))
    return candidates


def _refine_weights(
    matrices: Sequence[np.ndarray],
    labels: np.ndarray,
    weights: np.ndarray,
    mode: str,
    objective: str,
) -> np.ndarray:
    best_weights = weights.copy()
    best_score = _score_candidate(blend_predictions(matrices, best_weights, mode), labels, objective)

    for step_size in (0.10, 0.05, 0.02, 0.01, 0.005):
        improved = True
        while improved:
            improved = False
            for source in range(len(best_weights)):
                for target in range(len(best_weights)):
                    if source == target or best_weights[source] < step_size:
                        continue
                    candidate = best_weights.copy()
                    candidate[source] -= step_size
                    candidate[target] += step_size
                    score = _score_candidate(
                        blend_predictions(matrices, candidate, mode),
                        labels,
                        objective,
                    )
                    if score < best_score:
                        best_weights = candidate
                        best_score = score
                        improved = True
    return best_weights


def optimize_blend(
    predictions_by_name: Mapping[str, np.ndarray],
    labels: np.ndarray,
    objective: str = "log_loss",
    iterations: int = 4000,
    seed: int = 42,
) -> BlendResult:
    """Select weights and an averaging method from out-of-fold predictions."""

    if not predictions_by_name:
        raise ValueError("At least one model prediction set is required.")

    names = tuple(predictions_by_name)
    matrices = [normalize_probabilities(predictions_by_name[name]) for name in names]
    labels = np.asarray(labels, dtype=np.int64)

    best_score: tuple[float, float] | None = None
    best_weights: np.ndarray | None = None
    best_mode = ""

    for mode in ("probabilities", "logits"):
        for weights in _candidate_weights(len(names), iterations, seed):
            probabilities = blend_predictions(matrices, weights, mode)
            score = _score_candidate(probabilities, labels, objective)
            if best_score is None or score < best_score:
                best_score = score
                best_weights = np.asarray(weights, dtype=np.float64)
                best_mode = mode

    assert best_weights is not None
    best_weights = _refine_weights(matrices, labels, best_weights, best_mode, objective)
    final = blend_predictions(matrices, best_weights, best_mode)

    return BlendResult(
        names=names,
        weights=tuple(float(weight) for weight in best_weights / best_weights.sum()),
        mode=best_mode,
        accuracy=classification_accuracy(final, labels),
        log_loss=multiclass_log_loss(final, labels),
        objective=objective,
    )


### Dataset and Preprocessing


In [ ]:
"""Dataset discovery and model-aware image transformations."""


from collections import Counter
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
from PIL import Image
from timm.data import create_transform, resolve_model_data_config
from torch.utils.data import Dataset, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import InterpolationMode


IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def discover_train_items(train_dir: Path) -> list[tuple[Path, int]]:
    """Read class folders such as train/0, train/1, and so on."""

    items: list[tuple[Path, int]] = []
    class_dirs = [
        path
        for path in train_dir.iterdir()
        if path.is_dir() and path.name.isdigit()
    ]
    for class_dir in sorted(class_dirs, key=lambda path: int(path.name)):
        label = int(class_dir.name)
        for path in sorted(class_dir.iterdir()):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                items.append((path, label))
    return items


def discover_test_items(test_dir: Path) -> list[tuple[Path, str]]:
    def sort_key(path: Path) -> tuple[int, int | str]:
        return (0, int(path.stem)) if path.stem.isdigit() else (1, path.name)

    paths = [
        path
        for path in test_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ]
    return [(path, path.name) for path in sorted(paths, key=sort_key)]


class ImageDataset(Dataset):
    def __init__(self, items: Sequence[tuple[Path, object]], transform=None):
        self.items = items
        self.transform = transform

    def __len__(self) -> int:
        return len(self.items)

    def __getitem__(self, index: int):
        path, label_or_id = self.items[index]
        with Image.open(path) as image:
            image = image.convert("RGB")
        if self.transform is not None:
            image = self.transform(image)
        return image, label_or_id


def summarize_class_counts(labels: Sequence[int]) -> dict[int, int]:
    return dict(sorted(Counter(int(label) for label in labels).items()))


def make_balanced_sampler(
    labels: Sequence[int],
    automatically_balance: bool,
    imbalance_ratio_threshold: float,
) -> tuple[WeightedRandomSampler | None, float]:
    """Create a sampler only when class imbalance is substantial."""

    counts = Counter(int(label) for label in labels)
    imbalance_ratio = max(counts.values()) / min(counts.values())
    if not automatically_balance or imbalance_ratio < imbalance_ratio_threshold:
        return None, imbalance_ratio

    sample_weights = torch.tensor(
        [1.0 / counts[int(label)] for label in labels],
        dtype=torch.double,
    )
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )
    return sampler, imbalance_ratio


def _interpolation_mode(name: str) -> InterpolationMode:
    modes = {
        "bicubic": InterpolationMode.BICUBIC,
        "bilinear": InterpolationMode.BILINEAR,
        "nearest": InterpolationMode.NEAREST,
    }
    return modes.get(name, InterpolationMode.BICUBIC)


def make_transforms(model, variant: VariantConfig):
    """Use each pretrained backbone's normalization and interpolation settings."""

    data_config = resolve_model_data_config(model)
    interpolation = _interpolation_mode(data_config.get("interpolation", "bicubic"))
    mean = data_config["mean"]
    std = data_config["std"]

    train_transform = transforms.Compose(
        [
            transforms.RandomResizedCrop(
                variant.image_size,
                scale=(variant.crop_minimum, 1.0),
                interpolation=interpolation,
            ),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(variant.rotation_degrees),
            transforms.ColorJitter(
                brightness=variant.color_jitter_strength,
                contrast=variant.color_jitter_strength,
                saturation=variant.color_jitter_strength,
                hue=0.03,
            ),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
            transforms.RandomErasing(
                p=variant.random_erasing_probability,
                scale=(0.02, 0.18),
                ratio=(0.3, 3.3),
            ),
        ]
    )

    evaluation_config = dict(data_config)
    evaluation_config["input_size"] = (3, variant.image_size, variant.image_size)
    evaluation_transform = create_transform(**evaluation_config, is_training=False)

    return train_transform, evaluation_transform


### Model Creation


In [ ]:
"""Model resolution and optimizer construction."""


from typing import Iterable

import timm
import torch



def resolve_model_name(patterns: Iterable[str]) -> str:
    """Find the first installed timm pretrained model matching a preferred name."""

    available_models = set(timm.list_models(pretrained=True))
    for pattern in patterns:
        if pattern in available_models:
            return pattern
        matches = sorted(name for name in available_models if pattern in name)
        if matches:
            return matches[0]
    raise ValueError(f"No timm model matched: {tuple(patterns)}")


def create_model(
    model_name: str,
    variant: VariantConfig,
    num_classes: int,
    pretrained: bool,
):
    model = timm.create_model(
        model_name,
        pretrained=pretrained,
        num_classes=num_classes,
        drop_rate=variant.drop_rate,
        drop_path_rate=variant.drop_path_rate,
    )
    if pretrained and variant.gradient_checkpointing:
        set_grad_checkpointing = getattr(model, "set_grad_checkpointing", None)
        if set_grad_checkpointing is None:
            print(f"Gradient checkpointing is unavailable for {model_name}.")
        else:
            set_grad_checkpointing(enable=True)
    return model


def make_optimizer(model, variant: VariantConfig) -> torch.optim.Optimizer:
    """Fine-tune the backbone conservatively while adapting the classifier faster."""

    classifier = model.get_classifier()
    classifier_parameter_ids = {id(parameter) for parameter in classifier.parameters()}
    backbone_parameters = [
        parameter
        for parameter in model.parameters()
        if id(parameter) not in classifier_parameter_ids
    ]
    classifier_parameters = list(classifier.parameters())

    return torch.optim.AdamW(
        [
            {
                "params": backbone_parameters,
                "lr": variant.learning_rate * variant.backbone_lr_multiplier,
            },
            {
                "params": classifier_parameters,
                "lr": variant.learning_rate,
            },
        ],
        weight_decay=variant.weight_decay,
        foreach=False,
    )


### Training and Inference


In [ ]:
"""Training and inference for one model variant at a time."""


import gc
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Sequence

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import StratifiedKFold
from timm.data import Mixup
from timm.loss import SoftTargetCrossEntropy
from timm.utils import ModelEmaV2
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader



@dataclass
class VariantResult:
    name: str
    family: str
    model_name: str
    tta_mode: str
    oof_probabilities: np.ndarray
    test_probabilities: np.ndarray
    test_ids: list[str]
    fold_weights: np.ndarray
    fold_metrics: list[dict[str, object]]
    checkpoint_paths: list[str]

    def metadata(self) -> dict[str, object]:
        return {
            "name": self.name,
            "family": self.family,
            "model_name": self.model_name,
            "tta_mode": self.tta_mode,
            "oof_accuracy": classification_accuracy(self.oof_probabilities, self._labels),
            "oof_log_loss": multiclass_log_loss(self.oof_probabilities, self._labels),
            "fold_weights": self.fold_weights.tolist(),
            "fold_metrics": self.fold_metrics,
            "checkpoint_paths": self.checkpoint_paths,
        }

    # Labels are attached after construction to avoid storing them in saved metadata.
    _labels: np.ndarray | None = None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def cleanup(device: torch.device) -> None:
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def cuda_memory_summary(device: torch.device) -> str:
    if device.type != "cuda":
        return ""
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    peak = torch.cuda.max_memory_allocated() / 1024**3
    return f" | VRAM {allocated:.2f} GB allocated, {reserved:.2f} GB reserved, {peak:.2f} GB peak"


def make_scheduler(
    optimizer: torch.optim.Optimizer,
    total_steps: int,
    warmup_steps: int,
) -> torch.optim.lr_scheduler.LambdaLR:
    def multiplier(step: int) -> float:
        if step < warmup_steps:
            return float(step + 1) / float(max(1, warmup_steps))
        progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)


def _loader(
    dataset: ImageDataset,
    batch_size: int,
    num_workers: int,
    pin_memory: bool,
    persistent_workers: bool,
    prefetch_factor: int,
    shuffle: bool = False,
    sampler=None,
) -> DataLoader:
    worker_options = {}
    if num_workers > 0:
        worker_options = {
            "persistent_workers": persistent_workers,
            "prefetch_factor": prefetch_factor,
        }
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle if sampler is None else False,
        sampler=sampler,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
        **worker_options,
    )


def _train_one_epoch(
    model,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler,
    scaler: GradScaler,
    criterion,
    mixup: Mixup,
    accum_steps: int,
    ema: ModelEmaV2 | None,
    device: torch.device,
    use_amp: bool,
) -> tuple[float, float]:
    model.train()
    optimizer.zero_grad(set_to_none=True)
    total_loss = 0.0
    total_matches = 0
    total_samples = 0

    for step, (images, labels) in enumerate(loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        original_labels = labels.clone()
        images, mixed_labels = mixup(images, labels)

        with autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, mixed_labels) / accum_steps

        scaler.scale(loss).backward()
        should_update = (step + 1) % accum_steps == 0 or (step + 1) == len(loader)
        if should_update:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            if ema is not None:
                ema.update(model)

        batch_size = images.size(0)
        total_loss += loss.item() * accum_steps * batch_size
        total_matches += (logits.detach().argmax(dim=1) == original_labels).sum().item()
        total_samples += batch_size

    # This accuracy is only a rough diagnostic because Mixup/CutMix alter the inputs.
    return total_loss / total_samples, total_matches / total_samples


@torch.inference_mode()
def _predict_probabilities(
    model,
    loader: DataLoader,
    device: torch.device,
    use_amp: bool,
    tta_mode: str,
) -> tuple[np.ndarray, list[object]]:
    model.eval()
    all_probabilities: list[np.ndarray] = []
    all_values: list[object] = []

    for images, values in loader:
        images = images.to(device, non_blocking=True)
        with autocast(device_type=device.type, enabled=use_amp):
            logits = model(images)
            if tta_mode == "horizontal_flip":
                flipped_logits = model(torch.flip(images, dims=[3]))
                logits = (logits + flipped_logits) / 2.0
            elif tta_mode != "none":
                raise ValueError(f"Unsupported TTA mode: {tta_mode}")

        probabilities = torch.softmax(logits, dim=1).cpu().numpy()
        all_probabilities.append(probabilities)
        if isinstance(values, torch.Tensor):
            all_values.extend(values.cpu().numpy().tolist())
        else:
            all_values.extend(list(values))

    return normalize_probabilities(np.concatenate(all_probabilities)), all_values


def _evaluation_metrics(probabilities: np.ndarray, labels: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": classification_accuracy(probabilities, labels),
        "log_loss": multiclass_log_loss(probabilities, labels),
    }


def _save_checkpoint(
    path: Path,
    model,
    variant: VariantConfig,
    model_name: str,
    fold: int,
    epoch: int,
    metrics: dict[str, float],
) -> None:
    torch.save(
        {
            "variant": asdict(variant),
            "model_name": model_name,
            "fold": fold,
            "epoch": epoch,
            "metrics": metrics,
            "model_state_dict": model.state_dict(),
        },
        path,
    )


def _load_checkpoint_model(
    path: Path,
    model_name: str,
    variant: VariantConfig,
    config: ProjectConfig,
    device: torch.device,
):
    model = create_model(
        model_name=model_name,
        variant=variant,
        num_classes=config.num_classes,
        pretrained=False,
    )
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    del checkpoint
    return model.to(device)


def _choose_tta_mode(
    candidates: Sequence[str],
    oof_by_tta: dict[str, np.ndarray],
    labels: np.ndarray,
) -> tuple[str, dict[str, dict[str, float]]]:
    metrics = {
        mode: _evaluation_metrics(oof_by_tta[mode], labels)
        for mode in candidates
    }
    selected = min(
        candidates,
        key=lambda mode: (metrics[mode]["log_loss"], -metrics[mode]["accuracy"]),
    )
    return selected, metrics


def train_variant(
    variant: VariantConfig,
    config: ProjectConfig,
    train_items: Sequence[tuple[Path, int]],
    test_items: Sequence[tuple[Path, str]],
    labels: np.ndarray,
    device: torch.device,
) -> VariantResult:
    """Train folds, select TTA with OOF predictions, and infer one variant."""

    output_dir = config.output_dir / "variants" / variant.name
    output_dir.mkdir(parents=True, exist_ok=True)
    model_name = resolve_model_name(variant.model_patterns)
    print(f"\n{'=' * 88}\nTraining {variant.name}: {model_name}\n{'=' * 88}")

    template_model = create_model(model_name, variant, config.num_classes, pretrained=False)
    train_transform, evaluation_transform = make_transforms(template_model, variant)
    del template_model

    splitter = StratifiedKFold(
        n_splits=config.num_folds,
        shuffle=True,
        random_state=config.seed + variant.seed_offset,
    )
    oof_by_tta = {
        mode: np.zeros((len(train_items), config.num_classes), dtype=np.float32)
        for mode in config.tta_candidates
    }
    checkpoint_paths: list[str] = []
    fold_metrics: list[dict[str, object]] = []

    for fold, (train_indices, validation_indices) in enumerate(
        splitter.split(np.zeros(len(labels)), labels),
        start=1,
    ):
        fold_seed = config.seed + variant.seed_offset + fold
        seed_everything(fold_seed)
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        print(f"\n{variant.name} | fold {fold}/{config.num_folds} | seed {fold_seed}")

        fold_train_items = [train_items[index] for index in train_indices]
        fold_validation_items = [train_items[index] for index in validation_indices]
        fold_train_labels = labels[train_indices]
        fold_validation_labels = labels[validation_indices]
        sampler, imbalance_ratio = make_balanced_sampler(
            fold_train_labels,
            automatically_balance=config.automatically_balance_classes,
            imbalance_ratio_threshold=config.imbalance_ratio_threshold,
        )

        train_loader = _loader(
            ImageDataset(fold_train_items, train_transform),
            batch_size=variant.batch_size,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers,
            prefetch_factor=config.prefetch_factor,
            shuffle=True,
            sampler=sampler,
        )
        validation_loader = _loader(
            ImageDataset(fold_validation_items, evaluation_transform),
            batch_size=config.prediction_batch_size,
            num_workers=config.num_workers,
            pin_memory=config.pin_memory,
            persistent_workers=config.persistent_workers,
            prefetch_factor=config.prefetch_factor,
        )

        model = create_model(model_name, variant, config.num_classes, pretrained=True).to(device)
        ema = ModelEmaV2(model, decay=variant.ema_decay) if variant.use_ema else None
        optimizer = make_optimizer(model, variant)
        updates_per_epoch = math.ceil(len(train_loader) / variant.accum_steps)
        scheduler = make_scheduler(
            optimizer,
            total_steps=updates_per_epoch * config.final_epochs,
            warmup_steps=updates_per_epoch * variant.warmup_epochs,
        )
        scaler = GradScaler(device.type, enabled=config.use_amp and device.type == "cuda")
        mixup = Mixup(
            mixup_alpha=variant.mixup_alpha,
            cutmix_alpha=variant.cutmix_alpha,
            prob=variant.mixup_probability,
            switch_prob=0.5,
            mode="batch",
            label_smoothing=variant.label_smoothing,
            num_classes=config.num_classes,
        )
        criterion = SoftTargetCrossEntropy()

        checkpoint_path = output_dir / f"{variant.name}_fold{fold}_best.pt"
        best_metrics = {"accuracy": -1.0, "log_loss": float("inf")}
        best_epoch = -1
        stale_epochs = 0

        for epoch in range(1, config.final_epochs + 1):
            started = time.time()
            train_loss, mixed_input_accuracy = _train_one_epoch(
                model=model,
                loader=train_loader,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                criterion=criterion,
                mixup=mixup,
                accum_steps=variant.accum_steps,
                ema=ema,
                device=device,
                use_amp=config.use_amp and device.type == "cuda",
            )
            validation_model = ema.module if ema is not None else model
            validation_probabilities, _ = _predict_probabilities(
                validation_model,
                validation_loader,
                device,
                config.use_amp and device.type == "cuda",
                tta_mode="none",
            )
            validation_metrics = _evaluation_metrics(
                validation_probabilities,
                fold_validation_labels,
            )
            improved = (
                validation_metrics["accuracy"] > best_metrics["accuracy"]
                or (
                    validation_metrics["accuracy"] == best_metrics["accuracy"]
                    and validation_metrics["log_loss"] < best_metrics["log_loss"]
                )
            )
            if improved:
                best_metrics = validation_metrics
                best_epoch = epoch
                stale_epochs = 0
                _save_checkpoint(
                    checkpoint_path,
                    validation_model,
                    variant,
                    model_name,
                    fold,
                    epoch,
                    validation_metrics,
                )
            else:
                stale_epochs += 1

            print(
                f"{variant.name} | fold {fold} | epoch {epoch}/{config.final_epochs} | "
                f"train loss {train_loss:.4f} | mixed-input acc {mixed_input_accuracy:.4f} | "
                f"val loss {validation_metrics['log_loss']:.4f} | "
                f"val acc {validation_metrics['accuracy']:.4f} | "
                f"best {best_metrics['accuracy']:.4f} @ {best_epoch} | "
                f"stale {stale_epochs}/{variant.patience} | {time.time() - started:.1f}s"
                f"{cuda_memory_summary(device)}"
            )
            if stale_epochs >= variant.patience:
                print("Early stopping.")
                break

        del model, ema, optimizer, scheduler, scaler, train_loader
        cleanup(device)

        selected_model = _load_checkpoint_model(
            checkpoint_path,
            model_name,
            variant,
            config,
            device,
        )
        fold_tta_metrics: dict[str, dict[str, float]] = {}
        for tta_mode in config.tta_candidates:
            probabilities, _ = _predict_probabilities(
                selected_model,
                validation_loader,
                device,
                config.use_amp and device.type == "cuda",
                tta_mode=tta_mode,
            )
            oof_by_tta[tta_mode][validation_indices] = probabilities
            fold_tta_metrics[tta_mode] = _evaluation_metrics(
                probabilities,
                fold_validation_labels,
            )

        checkpoint_paths.append(str(checkpoint_path))
        fold_metrics.append(
            {
                "fold": fold,
                "seed": fold_seed,
                "checkpoint": str(checkpoint_path),
                "best_epoch": best_epoch,
                "checkpoint_metrics": best_metrics,
                "tta_metrics": fold_tta_metrics,
                "class_imbalance_ratio": imbalance_ratio,
                "used_balanced_sampler": sampler is not None,
            }
        )
        del selected_model, validation_loader
        cleanup(device)

    selected_tta, tta_metrics = _choose_tta_mode(config.tta_candidates, oof_by_tta, labels)
    selected_oof = oof_by_tta[selected_tta]
    selected_fold_losses = [
        float(metrics["tta_metrics"][selected_tta]["log_loss"])
        for metrics in fold_metrics
    ]
    fold_weights = fold_weights_from_losses(
        selected_fold_losses,
        temperature=config.fold_weight_temperature,
    )
    print(f"\n{variant.name} selected TTA: {selected_tta}")
    print(f"{variant.name} OOF TTA metrics: {tta_metrics}")
    print(f"{variant.name} fold weights: {fold_weights.tolist()}")

    test_loader = _loader(
        ImageDataset(test_items, evaluation_transform),
        batch_size=config.prediction_batch_size,
        num_workers=config.num_workers,
        pin_memory=config.pin_memory,
        persistent_workers=config.persistent_workers,
        prefetch_factor=config.prefetch_factor,
    )
    weighted_test_probabilities = np.zeros(
        (len(test_items), config.num_classes),
        dtype=np.float32,
    )
    test_ids: list[str] = []
    for checkpoint_path, fold_weight in zip(checkpoint_paths, fold_weights):
        selected_model = _load_checkpoint_model(
            Path(checkpoint_path),
            model_name,
            variant,
            config,
            device,
        )
        probabilities, fold_test_ids = _predict_probabilities(
            selected_model,
            test_loader,
            device,
            config.use_amp and device.type == "cuda",
            selected_tta,
        )
        if not test_ids:
            test_ids = [str(value) for value in fold_test_ids]
        elif test_ids != [str(value) for value in fold_test_ids]:
            raise ValueError("Test item order changed between fold predictions.")
        weighted_test_probabilities += probabilities * float(fold_weight)
        del selected_model
        cleanup(device)

    weighted_test_probabilities = normalize_probabilities(weighted_test_probabilities)
    np.save(output_dir / "oof_probabilities.npy", selected_oof)
    np.save(output_dir / "test_probabilities.npy", weighted_test_probabilities)
    with (output_dir / "variant_info.json").open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "name": variant.name,
                "family": variant.family,
                "model_name": model_name,
                "variant": asdict(variant),
                "selected_tta": selected_tta,
                "oof_tta_metrics": tta_metrics,
                "fold_weights": fold_weights.tolist(),
                "fold_metrics": fold_metrics,
                "checkpoint_paths": checkpoint_paths,
            },
            handle,
            indent=2,
        )

    result = VariantResult(
        name=variant.name,
        family=variant.family,
        model_name=model_name,
        tta_mode=selected_tta,
        oof_probabilities=selected_oof,
        test_probabilities=weighted_test_probabilities,
        test_ids=test_ids,
        fold_weights=fold_weights,
        fold_metrics=fold_metrics,
        checkpoint_paths=checkpoint_paths,
    )
    result._labels = labels
    del test_loader
    cleanup(device)
    return result


### Experiment Runner


In [ ]:
"""Run all enabled model variants and create the final submission."""


import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch



def run_experiment(config: ProjectConfig) -> Path:
    """Train variants, optimize the OOF blend, and write a Kaggle submission."""

    config.output_dir.mkdir(parents=True, exist_ok=True)
    train_items = discover_train_items(config.train_dir)
    test_items = discover_test_items(config.test_dir)
    labels = np.asarray([label for _, label in train_items], dtype=np.int64)
    class_counts = summarize_class_counts(labels)

    if not train_items:
        raise ValueError(f"No training images found under {config.train_dir}")
    if not test_items:
        raise ValueError(f"No test images found under {config.test_dir}")
    if len(class_counts) != config.num_classes:
        raise ValueError(
            f"Expected {config.num_classes} classes but discovered {len(class_counts)}."
        )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"GPU memory: {total_vram:.2f} GB")
    print(f"Training images: {len(train_items)}")
    print(f"Test images: {len(test_items)}")
    print(f"Smallest class: {min(class_counts.values())}")
    print(f"Largest class: {max(class_counts.values())}")

    results = []
    for variant in config.variants:
        if variant.enabled:
            results.append(
                train_variant(
                    variant=variant,
                    config=config,
                    train_items=train_items,
                    test_items=test_items,
                    labels=labels,
                    device=device,
                )
            )
    if not results:
        raise ValueError("No enabled model variants were configured.")

    oof_by_name = {result.name: result.oof_probabilities for result in results}
    test_by_name = {result.name: result.test_probabilities for result in results}
    blend = optimize_blend(
        oof_by_name,
        labels,
        objective=config.blend_objective,
        iterations=config.blend_search_iterations,
        seed=config.seed,
    )
    final_probabilities = blend_predictions(
        [test_by_name[name] for name in blend.names],
        blend.weights,
        blend.mode,
    )
    final_predictions = final_probabilities.argmax(axis=1).astype(int)
    test_ids = results[0].test_ids
    for result in results[1:]:
        if result.test_ids != test_ids:
            raise ValueError(f"Test ID mismatch for variant {result.name}.")

    submission_path = config.output_dir / "improved_oof_ensemble_ID_target.csv"
    probabilities_path = config.output_dir / "improved_oof_ensemble_probs.npy"
    info_path = config.output_dir / "improved_oof_ensemble_info.json"
    pd.DataFrame({"ID": test_ids, "target": final_predictions}).to_csv(
        submission_path,
        index=False,
    )
    np.save(probabilities_path, final_probabilities)
    with info_path.open("w", encoding="utf-8") as handle:
        json.dump(
            {
                "submission": str(submission_path),
                "probabilities": str(probabilities_path),
                "config": config.to_dict(),
                "class_counts": class_counts,
                "blend": blend.to_dict(),
                "variants": [result.metadata() for result in results],
            },
            handle,
            indent=2,
        )

    print(f"\nSelected blend: {blend.to_dict()}")
    print(f"Saved submission: {submission_path}")
    print(f"Saved probabilities: {probabilities_path}")
    print(f"Saved experiment metadata: {info_path}")
    return submission_path


## 5. Configure the Run

Leave `QUICK_RUN = False` for the full experiment. Set it to `True` only when
checking that the dataset paths and pipeline are wired correctly.

The defaults are intentionally memory-conscious: smaller micro-batches are
paired with gradient accumulation, EMA is opt-in, and DataLoader prefetching is
limited. This is slower than an aggressive H100-only setup but much less likely
to exhaust GPU or host memory.


In [ ]:
from dataclasses import replace
from pathlib import Path

KAGGLE_DIR = Path("/content/drive/MyDrive/Kaggle")
OUTPUT_DIR = KAGGLE_DIR / "improved_oof_ensemble"
QUICK_RUN = False

config = default_colab_config(KAGGLE_DIR)
config = replace(config, output_dir=OUTPUT_DIR)
if QUICK_RUN:
    config = replace(
        config,
        num_folds=2,
        final_epochs=2,
        blend_search_iterations=300,
    )

print("Train directory:", config.train_dir)
print("Test directory:", config.test_dir)
print("Output directory:", config.output_dir)
print("Variants:", [variant.name for variant in config.variants if variant.enabled])
for variant in config.variants:
    if variant.enabled:
        effective_batch = variant.batch_size * variant.accum_steps
        print(
            f"{variant.name}: micro-batch={variant.batch_size}, "
            f"accumulation={variant.accum_steps}, effective batch={effective_batch}, "
            f"gradient checkpointing={variant.gradient_checkpointing}, EMA={variant.use_ema}"
        )


## 6. Check Dataset Paths

This fast check catches missing Drive folders before GPU training begins.


In [ ]:
assert config.train_dir.exists(), f"Missing training directory: {config.train_dir}"
assert config.test_dir.exists(), f"Missing test directory: {config.test_dir}"

train_class_directories = [
    path for path in config.train_dir.iterdir()
    if path.is_dir() and path.name.isdigit()
]
test_images = [
    path for path in config.test_dir.iterdir()
    if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
]

print("Training class folders:", len(train_class_directories))
print("Test images:", len(test_images))
assert len(train_class_directories) == config.num_classes
assert test_images


## 7. Train and Create the Submission

This cell trains all enabled folds, generates out-of-fold predictions, selects
TTA behavior, learns ensemble weights, and writes the final CSV to Google Drive.


In [ ]:
submission_path = run_experiment(config)
print("Submission ready:", submission_path)
